# 01 ScenarioMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from cdrmip_extremes import (
    configs, load_pipeline, io_disk, gwl, utils,
    extremes_pipeline
)

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = load_pipeline.LoadConfig(
    experiments=[
        # load_pipeline.ExperimentConfig(
        #     experiment_id='ssp534-over', table_id='Amon', variable_id='tas',
        #     anomaly_reference='period', reference_period=(1850,1900),
        #     grid_label=None, check_end_year=True, threshold=2101,
        #     rolling_window=21, gwl_match_period = (2180,2200) 
        # ),
        load_pipeline.ExperimentConfig(
            experiment_id='ssp126', table_id='Amon', variable_id='tas',
            anomaly_reference='period', reference_period=(1850,1900),
            grid_label=None, check_end_year=True, threshold=2101,
            rolling_window=21, gwl_match_period = (2180,2200) 
        ),
    ],
    branch_date_overrides= {
        ('ACCESS-ESM1-5','ssp534-over'):'2041-01-01',   
    }
)

In [4]:
pipe = load_pipeline.LoadingPipeline(config)

In [5]:
pipe.run()

=== Loading ssp126 ===
Searching for files matching: {'experiment_id': 'ssp126', 'table_id': 'Amon', 'variable_id': 'tas', 'grid_label': None, 'member_id': None}
Loading ACCESS-CM2 / r10i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r1i1p1f1
  2015-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-CM2 / r2i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r3i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r4i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r5i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r6i1p1f1
  2015-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-CM2 / r7i1p1f1
  2015-01-16 12:00:00 ->

In [6]:
pipe.load_gsat().compute_gsat_rolling().compute_peak_warming()

In [7]:
pipe.peak_warming

{'ACCESS-CM2': {'ssp126': {'r1i1p1f1': {'peak': 2.6, 'peak_year': 2130}}},
 'ACCESS-ESM1-5': {'ssp126': {'r10i1p1f1': {'peak': 2.07, 'peak_year': 2066},
   'r11i1p1f1': {'peak': 1.98, 'peak_year': 2113},
   'r12i1p1f1': {'peak': 2.1, 'peak_year': 2085},
   'r13i1p1f1': {'peak': 2.12, 'peak_year': 2068},
   'r14i1p1f1': {'peak': 2.04, 'peak_year': 2104},
   'r15i1p1f1': {'peak': 2.11, 'peak_year': 2082},
   'r16i1p1f1': {'peak': 2.08, 'peak_year': 2128},
   'r17i1p1f1': {'peak': 2.02, 'peak_year': 2067},
   'r18i1p1f1': {'peak': 2.06, 'peak_year': 2098},
   'r19i1p1f1': {'peak': 2.13, 'peak_year': 2109},
   'r1i1p1f1': {'peak': 2.06, 'peak_year': 2079},
   'r20i1p1f1': {'peak': 2.09, 'peak_year': 2092},
   'r21i1p1f1': {'peak': 2.15, 'peak_year': 2073},
   'r22i1p1f1': {'peak': 2.14, 'peak_year': 2091},
   'r23i1p1f1': {'peak': 2.01, 'peak_year': 2071},
   'r24i1p1f1': {'peak': 2.05, 'peak_year': 2071},
   'r25i1p1f1': {'peak': 2.05, 'peak_year': 2084},
   'r26i1p1f1': {'peak': 2.13, 'p

In [8]:
pipe.compute_gwl_matching()

## Save Anomalies

In [9]:
pipe.save_tree(
    'anomaly',
    'tas',
    proc_dir = configs.PROC_DIR,
    activity_id = 'ScenarioMIP',
    category = 'anomaly'
)

pipe.save_tree(
    'gsat',
    'gsat',
    proc_dir = configs.PROC_DIR,
    activity_id = 'ScenarioMIP',
    category = None
)

pipe.save_tree(
    'gsat_rolling',
    'gsat_rolling',
    proc_dir = configs.PROC_DIR,
    activity_id = 'ScenarioMIP',
    category = None,
    # overwrite=True
)
pipe.save_tree(
    'gwl_matching',
    'gwl_matching_years',
    proc_dir = configs.PROC_DIR,
    activity_id = 'ScenarioMIP',
    category = None,
    # overwrite = True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-CM2/ssp126/ACCESS-CM2_ssp126_r1i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_r10i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_r11i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_r12i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_r13i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_r14i1p1f1_tas_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp

In [9]:
io_disk.save_peak_warming(pipe.peak_warming,'ScenarioMIP',configs.PROC_DIR)

## GWL Analysis

In [19]:
matching_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    pipe.anomaly, pipe.gwl_matching,
)

In [20]:
io_disk.save_tree(
    matching_diffs,
    configs.PROC_DIR,
    'tas_matching_gwl_differences',
    activity_id='ScenarioMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_r10i1p1f2_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_r11i1p1f2_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_r12i1p1f2_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_r13i1p1f2_tas_matching_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas_matching_gwl_differences/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_r14i1p1f2_tas_matching_gwl_differences.nc


## Extreme Temperature Frequency Analysis

In [12]:
ext = extremes_pipeline.ExtremesPipeline(
    regridded_piControl=pipe.regridded_piControl,
    regridded_concat=pipe.regridded_concat,
    gwl_matching=pipe.gwl_matching,
    gwl_window=21
)    

In [13]:
ext.run()

  No crossing_years provided — skipping gwl_period_extremes


In [14]:
ext.load(attribute='exceedances_matching')
ext.compute_multi_model_stat('exceedances_matching')

In [15]:
ext.compute_branch_differences(source_attr='exceedances_matching')
ext.compute_multi_model_stat('exceedances_matching')
ext.compute_multi_model_stat('branch_differences_matching')

In [17]:
ext.compute_agreement(source_attr='branch_differences_matching')

In [18]:
ext.save(
    configs.PROC_DIR,
    'ScenarioMIP',
    outputs=[
        'agreement_matching',
        'exceedances_matching_median_multimodel',
        'branch_differences_matching_median_multimodel'
    ],
    overwrite=False
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/agreement_matching/ssp534-over/ssp534-over_heat_tas_agreement_matching.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/agreement_matching/ssp534-over/ssp534-over_cold_tas_agreement_matching.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/exceedances_matching_median_multimodel/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_heat_tas_exceedances_matching_median_multimodel.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/exceedances_matching_median_multimodel/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_cold_tas_exceedances_matching_median_multimodel.nc
Skipping (already exists): /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/exceedances_matching_median_multimodel/CESM2-WACCM/ssp534-over/CESM2-WACCM_ssp534